# Week 6 - Model Pipeline Packaging and Inference Preparation

This notebook verifies that the final XGBoost fraud detection model can be
used outside the modeling notebook through a reusable inference pipeline.

The pipeline expects model-ready features: the 47 engineered features used
when training the final model. Raw transaction preprocessing is intentionally
kept outside this notebook because rolling and engineered features require a
separate feature pipeline.

## 1. Import Libraries

The notebook imports the reusable pipeline from `src/inference_pipeline.py`.
The `sys.path` update allows this notebook to import project modules from the
repository root.

In [1]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.append("..")

from src.inference_pipeline import FraudInferencePipeline

REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(exist_ok=True)

## 2. Load Test Data

The test data is loaded only to provide model-ready sample transactions for
pipeline validation. The `Class` column is kept separately as the actual label.

In [2]:
test_df = pd.read_csv("../data/processed/test_fe.csv")

X_test = test_df.drop(columns=["Class"])
y_test = test_df["Class"]

test_data_summary = pd.DataFrame(
    {
        "rows": [len(X_test)],
        "features": [X_test.shape[1]],
        "fraud_count": [int(y_test.sum())],
        "non_fraud_count": [int((y_test == 0).sum())],
    }
)

test_data_summary

,rows,features,fraud_count,non_fraud_count
0,56962,47,75,56887


## 3. Initialize Inference Pipeline

The pipeline loads artifacts from `models/` and stores the selected threshold
and ordered feature list. Expected values are threshold `0.73` and `47`
features.

In [3]:
pipeline = FraudInferencePipeline()

pipeline_info = pd.DataFrame(
    [
        {
            "threshold": pipeline.threshold,
            "n_features": len(pipeline.feature_list),
            "first_10_features": pipeline.feature_list[:10],
        }
    ]
)

pipeline_info

,threshold,n_features,first_10_features
0,0.73,47,"[Time, V1, V2, V3, V4, V5, V6, V7, V8, V9]"


## 4. Single Transaction Prediction

This cell tests the smallest unit of inference: one transaction represented as
a Python dictionary. The same JSON-style payload will be reused by FastAPI in
Week 7.

In [4]:
sample_transaction = X_test.iloc[0].to_dict()

with open(
    REPORT_DIR / "sample_transaction_payload.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(sample_transaction, file, indent=4)

single_prediction = pipeline.predict(sample_transaction)
single_prediction.to_csv(
    REPORT_DIR / "week6_single_prediction_test.csv",
    index=False,
)

single_prediction

,fraud_probability,prediction,prediction_label,threshold
0,0.000008,0,Non-fraud,0.73


### Interpretation

The output includes the fraud probability, binary prediction, label, and
threshold. This is the same structure that can be returned by a future API
endpoint.

## 5. Batch Prediction on 100 Transactions

This test confirms that the same pipeline works for multiple transactions in a
DataFrame. The output is combined with actual labels for quick inspection.

In [5]:
batch_sample = X_test.head(100)
batch_prediction = pipeline.predict(batch_sample)

batch_output = batch_sample.copy()
batch_output["actual"] = y_test.head(100).values
batch_output["fraud_probability"] = batch_prediction[
    "fraud_probability"
].values
batch_output["prediction"] = batch_prediction["prediction"].values
batch_output["prediction_label"] = batch_prediction[
    "prediction_label"
].values

batch_output.to_csv(
    REPORT_DIR / "week6_batch_prediction_sample.csv",
    index=False,
)

full_batch_output_path = REPORT_DIR / "week6_batch_prediction_test.csv"

if not full_batch_output_path.exists():
    full_batch_prediction = pipeline.predict(X_test)
    full_batch_output = test_df.copy()
    full_batch_output["fraud_probability"] = full_batch_prediction[
        "fraud_probability"
    ].values
    full_batch_output["prediction"] = full_batch_prediction[
        "prediction"
    ].values
    full_batch_output["prediction_label"] = full_batch_prediction[
        "prediction_label"
    ].values
    full_batch_output.to_csv(full_batch_output_path, index=False)

batch_output.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,amount_roll_median_10,amount_dev_median_10,tx_count_proxy_5,tx_count_proxy_10,amount_time_interaction,log_amount_hour_interaction,actual,fraud_probability,prediction,prediction_label
0,145248.0,1.914027,-0.490068,-0.326111,0.604711,-0.850136,-0.736319,-0.524058,-0.088614,1.091125,...,24.805,25.195,5.0,10.0,25.000,62.909210,0,0.000008,0,Non-fraud
1,145249.0,2.152696,-0.036161,-2.231811,0.091766,0.537612,-1.368103,0.613327,-0.455252,0.291814,...,19.805,-4.855,5.0,10.0,7.475,44.311341,0,0.000006,0,Non-fraud
2,145249.0,-4.034795,2.305079,-1.461693,-0.729887,-1.528750,-1.225679,-0.893354,1.622522,1.291998,...,19.805,-12.105,5.0,10.0,7.700,34.613168,0,0.000002,0,Non-fraud
3,145249.0,-1.668741,1.168055,0.249642,-1.268497,0.785923,-0.663959,0.859433,0.068111,-0.144183,...,11.325,-4.335,5.0,10.0,6.990,33.251052,0,0.000030,0,Non-fraud
4,145250.0,1.846910,0.143301,-1.171846,1.570946,0.076854,-0.858130,0.164378,-0.251494,0.442113,...,11.325,56.675,5.0,10.0,34.000,67.745704,0,0.000578,0,Non-fraud


## 6. Compare Pipeline Prediction with Manual Week 4 Logic

This check verifies that the packaged pipeline produces the same probabilities
and labels as direct model inference. The expected result is max probability
difference `0.0` and prediction match `True`.

In [6]:
model = joblib.load("../models/final_fraud_model.pkl")
threshold = float(joblib.load("../models/best_threshold.pkl"))
feature_list = joblib.load("../models/feature_list.pkl")

X_test_aligned = X_test[feature_list]

manual_prob = model.predict_proba(X_test_aligned.head(100))[:, 1]
manual_pred = (manual_prob >= threshold).astype(int)

pipeline_prob = batch_prediction["fraud_probability"].values
pipeline_pred = batch_prediction["prediction"].values

comparison_result = pd.DataFrame(
    [
        {
            "probability_difference_max": np.max(
                np.abs(manual_prob - pipeline_prob)
            ),
            "prediction_match": np.array_equal(
                manual_pred,
                pipeline_pred,
            ),
        }
    ]
)

comparison_result.to_csv(
    REPORT_DIR / "week6_manual_pipeline_comparison.csv",
    index=False,
)

comparison_result

,probability_difference_max,prediction_match
0,0.0,True


## 7. Missing Feature Validation Test

A production-facing pipeline must reject invalid input. This cell intentionally
removes one feature to confirm that the pipeline raises a clear validation
error instead of silently producing unreliable predictions.

In [7]:
bad_sample = X_test.iloc[0].drop(
    labels=[pipeline.feature_list[0]]
).to_dict()

try:
    pipeline.predict(bad_sample)
except ValueError as error:
    validation_error_message = str(error)
    print("Validation error detected:")
    print(validation_error_message)

Validation error detected:
Missing required features: ['Time']


## 8. Latency Test

This lightweight test measures repeated single-transaction inference. It does
not represent a production benchmark, but it shows that the model can support a
near real-time prototype.

In [8]:
latencies = []
n_runs = 1000

for _ in range(n_runs):
    start = time.perf_counter()
    pipeline.predict(sample_transaction)
    end = time.perf_counter()
    latencies.append((end - start) * 1000)

latency_summary = pd.DataFrame(
    [
        {
            "n_runs": n_runs,
            "mean_latency_ms": np.mean(latencies),
            "median_latency_ms": np.median(latencies),
            "p95_latency_ms": np.percentile(latencies, 95),
            "p99_latency_ms": np.percentile(latencies, 99),
            "min_latency_ms": np.min(latencies),
            "max_latency_ms": np.max(latencies),
        }
    ]
)

# Keep the original filename for compatibility, and save the richer summary
# under a clearer Week 6 report name.
latency_result = latency_summary.copy()
latency_summary.to_csv(
    REPORT_DIR / "week6_latency_summary.csv",
    index=False,
)
latency_result.to_csv(
    REPORT_DIR / "week6_latency_test.csv",
    index=False,
)

latency_summary

,n_runs,mean_latency_ms,median_latency_ms,p95_latency_ms,p99_latency_ms,min_latency_ms,max_latency_ms
0,1000,31.947467,30.39555,38.742525,53.118169,25.1654,195.4074


## 8.1. Batch Latency Test

Single-transaction latency is useful for real-time scoring, but the system may also process small batches. This test compares total latency and average latency per transaction for batch sizes `1`, `10`, `100`, and `1000`.

In [9]:
batch_sizes = [1, 10, 100, 1000]
batch_latency_results = []

for size in batch_sizes:
    batch = X_test.head(size)

    start = time.perf_counter()
    prediction = pipeline.predict(batch)
    end = time.perf_counter()

    total_ms = (end - start) * 1000
    avg_per_tx_ms = total_ms / size

    batch_latency_results.append(
        {
            "batch_size": size,
            "total_latency_ms": total_ms,
            "avg_latency_per_transaction_ms": avg_per_tx_ms,
            "n_predictions": len(prediction),
        }
    )

batch_latency_df = pd.DataFrame(batch_latency_results)
batch_latency_df.to_csv(
    REPORT_DIR / "week6_batch_latency_summary.csv",
    index=False,
)

batch_latency_df

,batch_size,total_latency_ms,avg_latency_per_transaction_ms,n_predictions
0,1,29.1812,29.181200,1
1,10,30.4915,3.049150,10
2,100,38.2926,0.382926,100
3,1000,42.9987,0.042999,1000


## 9. Simulate Transaction Stream

This is a simple stream simulation where transactions are processed one by one.
It is not FastAPI yet, but it mirrors the logic needed for Week 7.

In [10]:
stream_sample = X_test.head(20).copy()
stream_results = []

for idx, row in stream_sample.iterrows():
    transaction = row.to_dict()
    prediction = pipeline.predict(transaction)

    stream_results.append(
        {
            "transaction_index": idx,
            "fraud_probability": prediction[
                "fraud_probability"
            ].iloc[0],
            "prediction": prediction["prediction"].iloc[0],
            "prediction_label": prediction[
                "prediction_label"
            ].iloc[0],
            "threshold": prediction["threshold"].iloc[0],
        }
    )

stream_results_df = pd.DataFrame(stream_results)
stream_results_df.to_csv(
    REPORT_DIR / "week6_stream_simulation.csv",
    index=False,
)

stream_results_df.head()

,transaction_index,fraud_probability,prediction,prediction_label,threshold
0,0,0.000008,0,Non-fraud,0.73
1,1,0.000006,0,Non-fraud,0.73
2,2,0.000002,0,Non-fraud,0.73
3,3,0.000030,0,Non-fraud,0.73
4,4,0.000578,0,Non-fraud,0.73


## 9.1. Mixed Stream Simulation for Demo

The first stream simulation uses the first rows of the test set, which are all non-fraud. For a stronger fraud detection demo, this mixed stream includes:

- correctly detected fraud transactions,
- normal transactions incorrectly flagged as fraud,
- normal non-fraud transactions.

This better demonstrates how the pipeline behaves when fraud and non-fraud transactions arrive in the same simulated stream.

In [11]:
all_test_predictions = pipeline.predict(X_test)

prediction_analysis_df = X_test.copy()
prediction_analysis_df["actual"] = y_test.values
prediction_analysis_df["prediction"] = all_test_predictions[
    "prediction"
].values
prediction_analysis_df["fraud_probability"] = all_test_predictions[
    "fraud_probability"
].values

true_positive_features = prediction_analysis_df[
    (prediction_analysis_df["actual"] == 1)
    & (prediction_analysis_df["prediction"] == 1)
].drop(
    columns=["actual", "prediction", "fraud_probability"],
    errors="ignore",
).head(3)

false_positive_features = prediction_analysis_df[
    (prediction_analysis_df["actual"] == 0)
    & (prediction_analysis_df["prediction"] == 1)
].drop(
    columns=["actual", "prediction", "fraud_probability"],
    errors="ignore",
).head(2)

normal_features = prediction_analysis_df[
    (prediction_analysis_df["actual"] == 0)
    & (prediction_analysis_df["prediction"] == 0)
].drop(
    columns=["actual", "prediction", "fraud_probability"],
    errors="ignore",
).head(5)

mixed_stream_df = pd.concat(
    [
        true_positive_features,
        false_positive_features,
        normal_features,
    ]
)

stream_results = []

for idx, row in mixed_stream_df.iterrows():
    transaction = row.to_dict()
    prediction = pipeline.predict(transaction)

    stream_results.append(
        {
            "transaction_index": idx,
            "fraud_probability": prediction[
                "fraud_probability"
            ].iloc[0],
            "prediction": prediction["prediction"].iloc[0],
            "prediction_label": prediction[
                "prediction_label"
            ].iloc[0],
            "threshold": prediction["threshold"].iloc[0],
            "actual": int(y_test.loc[idx]),
        }
    )

mixed_stream_results_df = pd.DataFrame(stream_results)
mixed_stream_results_df.to_csv(
    REPORT_DIR / "week6_stream_simulation_mixed.csv",
    index=False,
)

mixed_stream_results_df

,transaction_index,fraud_probability,prediction,prediction_label,threshold,actual
0,1866,0.999982,1,Fraud,0.73,1
1,2231,0.999899,1,Fraud,0.73,1
2,2631,0.999792,1,Fraud,0.73,1
3,659,0.988036,1,Fraud,0.73,0
4,9346,0.766567,1,Fraud,0.73,0
5,0,0.000008,0,Non-fraud,0.73,0
6,1,0.000006,0,Non-fraud,0.73,0
7,2,0.000002,0,Non-fraud,0.73,0
8,3,0.000030,0,Non-fraud,0.73,0
9,4,0.000578,0,Non-fraud,0.73,0


### Mixed Stream Demo Interpretation

This table is better suited for a presentation than a stream containing only non-fraud rows. It shows the same inference pipeline processing both normal and fraudulent transactions, then returning fraud probability and final decision using the selected threshold of `0.73`.

## 10. Prepare API-Like Output Example

FastAPI should return JSON-serializable values. This cell converts the single
prediction result into a clean dictionary and saves it for Week 7 API testing.

In [12]:
api_like_output = single_prediction.iloc[0].to_dict()
api_like_output = {
    "fraud_probability": float(api_like_output["fraud_probability"]),
    "prediction": int(api_like_output["prediction"]),
    "prediction_label": str(api_like_output["prediction_label"]),
    "threshold": float(api_like_output["threshold"]),
}

with open(
    REPORT_DIR / "week6_api_output_example.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(api_like_output, file, indent=4)

api_like_output

{'fraud_probability': 8.364269888261333e-06,
 'prediction': 0,
 'prediction_label': 'Non-fraud',
 'threshold': 0.73}

## 10.1. Prepare Audit Log Format

A fraud detection system should keep a traceable audit record for each prediction. This example log records prediction time, transaction id, model version, fraud probability, prediction, label, and threshold.

The file is saved as `reports/week6_audit_log_example.json` for Week 7 API and dashboard preparation.

In [13]:
from datetime import datetime, timezone

model_version = "xgboost_v1_threshold_0.73"

audit_log_example = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "transaction_id": "sample_tx_001",
    "model_version": model_version,
    "fraud_probability": float(
        single_prediction["fraud_probability"].iloc[0]
    ),
    "prediction": int(single_prediction["prediction"].iloc[0]),
    "prediction_label": str(
        single_prediction["prediction_label"].iloc[0]
    ),
    "threshold": float(single_prediction["threshold"].iloc[0]),
}

with open(
    REPORT_DIR / "week6_audit_log_example.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(audit_log_example, file, indent=4)

audit_log_example

{'timestamp': '2026-06-08T13:35:08.051523+00:00',
 'transaction_id': 'sample_tx_001',
 'model_version': 'xgboost_v1_threshold_0.73',
 'fraud_probability': 8.364269888261333e-06,
 'prediction': 0,
 'prediction_label': 'Non-fraud',
 'threshold': 0.73}

## 11. Week 6 Output File Check

This final cell verifies that the expected Week 6 files were created. These
files will be used as evidence for the pipeline packaging stage and as inputs
for Week 7 API development.

In [14]:
required_week6_outputs = [
    "sample_transaction_payload.json",
    "week6_single_prediction_test.csv",
    "week6_batch_prediction_sample.csv",
    "week6_batch_prediction_test.csv",
    "week6_manual_pipeline_comparison.csv",
    "week6_latency_test.csv",
    "week6_batch_latency_summary.csv",
    "week6_latency_summary.csv",
    "week6_stream_simulation.csv",
    "week6_stream_simulation_mixed.csv",
    "week6_api_output_example.json",
    "week6_audit_log_example.json",
    "week6_src_file_status.csv",
    "week6_submission_file_inventory.csv",
    "week6_input_schema.json",
    "model_card.md",
    "week6_pipeline_notes.md",
]

week6_output_check = pd.DataFrame(
    [
        {
            "file": file,
            "exists": (REPORT_DIR / file).exists(),
            "size_bytes": (
                REPORT_DIR / file
            ).stat().st_size
            if (REPORT_DIR / file).exists()
            else 0,
        }
        for file in required_week6_outputs
    ]
)

print(
    "All required Week 6 outputs present:",
    week6_output_check["exists"].all(),
)

week6_output_check

All required Week 6 outputs present: True


,file,exists,size_bytes
0,sample_transaction_payload.json,True,1552
1,week6_single_prediction_test.csv,True,87
2,week6_batch_prediction_sample.csv,True,76532
3,week6_batch_prediction_test.csv,True,43862583
4,week6_manual_pipeline_comparison.csv,True,55
5,week6_latency_test.csv,True,220
6,week6_batch_latency_summary.csv,True,257
7,week6_latency_summary.csv,True,220
8,week6_stream_simulation.csv,True,757
9,week6_stream_simulation_mixed.csv,True,413


## Summary of Week 6 Pipeline Packaging

The final XGBoost model has been packaged into a reusable inference pipeline.
The pipeline loads the trained model, selected threshold, and feature list;
validates incoming model-ready features; aligns feature order; handles missing
or infinite values; computes fraud probability; and returns a final decision
using the selected threshold of `0.73`.

The notebook confirmed that single prediction and batch prediction work, that
pipeline output matches direct manual model inference, and that invalid input is
caught through feature validation. Single-transaction latency, batch latency, basic stream simulation, mixed stream simulation, and audit log format tests were created to prepare for a simulated real-time fraud detection API in Week 7.

## Week 6 Submission Evidence

This final section collects the six requested outputs for Week 6 review. The code cells below are designed to be screenshot-ready inside the notebook.

### 1. Screenshot Evidence: `python -m src.validate_artifacts`

The output below is equivalent to running `python -m src.validate_artifacts` from the project root. It confirms that the saved model, threshold, and feature list can be loaded correctly.

In [15]:
import subprocess

validate_artifacts_result = subprocess.run(
    [sys.executable, "-m", "src.validate_artifacts"],
    cwd=Path(".."),
    capture_output=True,
    text=True,
    check=False,
)

print(validate_artifacts_result.stdout)

if validate_artifacts_result.stderr:
    print("STDERR:")
    print(validate_artifacts_result.stderr)

print("Return code:", validate_artifacts_result.returncode)

Artifact validation successful.
Model type: <class 'xgboost.sklearn.XGBClassifier'>
Threshold: 0.73
Number of features: 47
First 10 features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']

Return code: 0


### 2. Output: `single_prediction`

This is the prediction result for one model-ready transaction. It is the expected response shape for a future API endpoint.

In [16]:
single_prediction

,fraud_probability,prediction,prediction_label,threshold
0,0.000008,0,Non-fraud,0.73


### 3. Output: Manual Prediction vs Pipeline Prediction

This confirms that the packaged pipeline produces the same probabilities and predictions as the direct Week 4 model inference logic.

In [17]:
comparison_result

,probability_difference_max,prediction_match
0,0.0,True


### 4. Output: Single-Transaction Latency Summary

This table reports mean, median/p50, p95, p99, min, and max latency for repeated single-transaction predictions.

In [18]:
latency_summary

,n_runs,mean_latency_ms,median_latency_ms,p95_latency_ms,p99_latency_ms,min_latency_ms,max_latency_ms
0,1000,31.947467,30.39555,38.742525,53.118169,25.1654,195.4074


### 4.1. Output: Batch Latency Summary

This table shows total batch latency and average latency per transaction, which helps discuss throughput and batch inference behavior.

In [19]:
batch_latency_df

,batch_size,total_latency_ms,avg_latency_per_transaction_ms,n_predictions
0,1,29.1812,29.181200,1
1,10,30.4915,3.049150,10
2,100,38.2926,0.382926,100
3,1000,42.9987,0.042999,1000


### 5. Screenshot Evidence: `stream_results_df.head()`

This shows a simple transaction-stream simulation where transactions are processed one by one.

In [20]:
stream_results_df.head()

,transaction_index,fraud_probability,prediction,prediction_label,threshold
0,0,0.000008,0,Non-fraud,0.73
1,1,0.000006,0,Non-fraud,0.73
2,2,0.000002,0,Non-fraud,0.73
3,3,0.000030,0,Non-fraud,0.73
4,4,0.000578,0,Non-fraud,0.73


### 5.1. Screenshot Evidence: `mixed_stream_results_df`

This mixed stream is recommended for demo because it contains both fraud and non-fraud transactions.

In [21]:
mixed_stream_results_df

,transaction_index,fraud_probability,prediction,prediction_label,threshold,actual
0,1866,0.999982,1,Fraud,0.73,1
1,2231,0.999899,1,Fraud,0.73,1
2,2631,0.999792,1,Fraud,0.73,1
3,659,0.988036,1,Fraud,0.73,0
4,9346,0.766567,1,Fraud,0.73,0
5,0,0.000008,0,Non-fraud,0.73,0
6,1,0.000006,0,Non-fraud,0.73,0
7,2,0.000002,0,Non-fraud,0.73,0
8,3,0.000030,0,Non-fraud,0.73,0
9,4,0.000578,0,Non-fraud,0.73,0


### 6. Audit Log Example

This is the traceability record to show what the system would store when a prediction is made.

In [22]:
audit_log_example

{'timestamp': '2026-06-08T13:35:08.051523+00:00',
 'transaction_id': 'sample_tx_001',
 'model_version': 'xgboost_v1_threshold_0.73',
 'fraud_probability': 8.364269888261333e-06,
 'prediction': 0,
 'prediction_label': 'Non-fraud',
 'threshold': 0.73}

### 7. Source Code Scope Review

This table separates the active Week 6 inference code from utility entrypoints and placeholder modules. It prevents older skeleton files from being mistaken for completed production logic.

In [23]:
src_file_status = pd.read_csv(REPORT_DIR / "week6_src_file_status.csv")
src_file_status

,file,status,role,notes
0,src/model_loader.py,active,artifact_loader,"Loads final_fraud_model.pkl, best_threshold.pk..."
1,src/inference_pipeline.py,active,inference_pipeline,"Validates and aligns model-ready features, the..."
2,src/batch_inference.py,utility,batch_inference,Runs CSV batch inference using FraudInferenceP...
3,src/validate_artifacts.py,utility,artifact_validation,Checks that saved model artifacts can be loade...
4,src/data_preprocessing.py,placeholder,data_preprocessing,Not used by Week 6. Data preparation remains i...
5,src/feature_engineering.py,placeholder,feature_engineering,Not used by Week 6. The current model expects ...
6,src/train_model.py,placeholder,training,Not used by Week 6. Final training is document...
7,src/evaluate_model.py,placeholder,evaluation,Not used by Week 6. Evaluation outputs are gen...
8,src/explainability.py,placeholder,xai,Not used by Week 6. SHAP/LIME outputs are gene...
9,src/inference_api.py,placeholder,api,Not implemented yet. Planned for Week 7 FastAP...
